# MarketingIQ tiny planner — Phase 5: final frozen evaluation (Qwen3-0.6B + Phase 4 LoRA)

**The one-time, unbiased exam.** The Phase 4 model answers the 429 questions of `eval_frozen.jsonl`
(never used for training, validation, checkpoint choice or prompt tuning). Each output goes through:

`raw output → parse → plan_format.to_canonical() → validate_plan() → execute_plan() on DuckDB`

and is compared with the gold plan by the Phase 2 scorer `evaluation/score.py` (exact plan match and
execution match). Every stage is recorded separately; nothing is repaired.

**Rules for this phase**
- **No training.** No `trainer`, no optimizer, no weight update. The adapter files are SHA-256 hashed before
  and after, and the live LoRA weights are fingerprinted before and after generation.
- The Phase 4 adapter is used as saved (final model after epoch 2; no checkpoint selection, no merge, no quantization).
- Same prompt as Phase 4 (`plan_format.messages()` + Qwen3 chat template, **thinking OFF**), greedy decoding,
  same max-new-tokens / EOS / padding as Phase 4 validation.
- The results are final: do not change the prompt, settings or model because of them.

Step-by-step instructions: `experiments/tiny_model/PHASE5_INSTRUCTIONS.md`.

## 0. Configuration

In [ ]:
MODEL_ID = "unsloth/Qwen3-0.6B"                                    # Phase 4 base model

DRIVE_DIR       = "/content/drive/MyDrive/marketingiq_tiny"
EVAL_BUNDLE_ZIP = f"{DRIVE_DIR}/marketingiq_eval_bundle.zip"         # made by notebooks/make_eval_bundle.py
RUN_NAME        = "marketingiq-qwen3-0.6b-lora"                      # Phase 4 run folder suffix
PHASE4_RUN      = None   # folder name under runs/, e.g. "20261001-101500_marketingiq-qwen3-0.6b-lora";
                         # None = the only Phase 4 run there. Choose it from Phase 4 records, never from these results.
WORK_DIR   = "/content/marketingiq_eval"                             # bundle is unzipped here (temporary)
OUTPUT_DIR = "/content/phase5_outputs"                               # results (temporary, copied to Drive)

FROZEN_SHA256 = "51966831e505de8ac59fec49dad53cb1c2d4daa8744fa46d75794be831c7f2b1"
FROZEN_COUNT  = 429

GEN_BATCH_SIZE = 32      # Phase 4 validation inference setting
SEED = 20260929          # Phase 4 seed (greedy decoding does not sample; set for completeness)

## 1. Environment Setup
In Colab: **Runtime → Change runtime type → T4 GPU** (the GPU Phase 4 trained on, if possible).

In [ ]:
import sys, subprocess, platform
print("Python", sys.version.split()[0], "on", platform.platform())

def stop(message):
    raise RuntimeError("\n\n" + "=" * 70 + "\nSTOPPED: " + message + "\n" + "=" * 70)

import torch
if not torch.cuda.is_available():
    stop("No CUDA GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart and run again.")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
props = torch.cuda.get_device_properties(0)
GPU_NAME, GPU_VRAM_GB = props.name, round(props.total_memory / 1024**3, 1)
print(f"GPU: {GPU_NAME} | VRAM: {GPU_VRAM_GB} GB | compute capability {props.major}.{props.minor}")
if GPU_VRAM_GB < 12:
    print("WARNING: less than 12 GB VRAM; lower BATCH_SIZE to 4 and raise GRAD_ACCUM to 4 in section 0.")

Install Unsloth (identical to the Phase 4 install cell). This is the Colab install cell from Unsloth's official Qwen3 notebooks
(github.com/unslothai/notebooks, checked Sept 2026: `transformers==4.56.2`, `trl==0.22.2`), plus `duckdb`,
which the MarketingIQ validator/executor needs. Takes 1–3 minutes.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install duckdb

In [ ]:
import unsloth  # import before transformers so Unsloth's patches apply
import importlib.metadata as md_
VERSIONS = {p: md_.version(p) for p in ["unsloth", "unsloth_zoo", "torch", "transformers", "trl", "peft",
                                        "datasets", "accelerate", "bitsandbytes", "duckdb", "pandas"]}
VERSIONS["python"] = sys.version.split()[0]
for k, v in VERSIONS.items():
    print(f"{k:14s} {v}")

## 2. Phase 4 model (Google Drive)
Finds the Phase 4 run saved by the training notebook (`MyDrive/marketingiq_tiny/runs/<date-time>_marketingiq-qwen3-0.6b-lora/`),
reads its `training_report.json` and hashes every adapter file.

In [ ]:
import os, json, hashlib, shutil, zipfile, datetime, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def hash_dir(d):
    return {str(p.relative_to(d)): sha256_file(p) for p in sorted(Path(d).rglob("*")) if p.is_file()}

runs_dir = Path(DRIVE_DIR) / "runs"
if PHASE4_RUN:
    RUN_DIR = runs_dir / PHASE4_RUN
else:
    found = sorted(p for p in runs_dir.glob(f"*_{RUN_NAME}") if (p / "adapter").is_dir())
    if len(found) != 1:
        stop(f"{len(found)} Phase 4 runs found in {runs_dir}: {[p.name for p in found]}. Set PHASE4_RUN in section 0 "
             "to the run Phase 4 produced (decide from Phase 4 records only, never from frozen-eval results).")
    RUN_DIR = found[0]
ADAPTER_DIR = RUN_DIR / "adapter"
for f in ("adapter_config.json", "adapter_model.safetensors", "marketingiq_prompt_contract.json"):
    if not (ADAPTER_DIR / f).is_file():
        stop(f"{ADAPTER_DIR / f} is missing; this is not a complete Phase 4 adapter folder.")
if not (RUN_DIR / "training_report.json").is_file():
    stop(f"{RUN_DIR / 'training_report.json'} is missing.")

TRAIN_REPORT = json.loads((RUN_DIR / "training_report.json").read_text())
CONTRACT = json.loads((ADAPTER_DIR / "marketingiq_prompt_contract.json").read_text())
ADAPTER_CONFIG = json.loads((ADAPTER_DIR / "adapter_config.json").read_text())
ADAPTER_SHA_BEFORE = hash_dir(ADAPTER_DIR)

problems = []
if TRAIN_REPORT.get("phase") != 4: problems.append("training_report.json is not a Phase 4 report")
if TRAIN_REPORT.get("model_id") != MODEL_ID: problems.append(f"Phase 4 model_id {TRAIN_REPORT.get('model_id')!r} != {MODEL_ID!r}")
if TRAIN_REPORT.get("frozen_eval_used") is not False: problems.append("Phase 4 report does not confirm frozen_eval_used = false")
if TRAIN_REPORT.get("thinking_mode") != "OFF": problems.append("Phase 4 was not trained with thinking OFF")
if CONTRACT != TRAIN_REPORT.get("prompt_contract"): problems.append("adapter prompt contract differs from training_report.json")
if str(ADAPTER_CONFIG.get("base_model_name_or_path", "")).lower() != MODEL_ID.lower():
    problems.append(f"adapter base model {ADAPTER_CONFIG.get('base_model_name_or_path')!r} != {MODEL_ID!r}")
if problems:
    stop("Phase 4 artifacts do not match:\n" + "\n".join(problems))

MAX_SEQ_LENGTH, MAX_NEW_TOKENS = CONTRACT["max_seq_length"], CONTRACT["max_new_tokens"]
print("Phase 4 run:      ", RUN_DIR.name)
print("Base model:       ", ADAPTER_CONFIG["base_model_name_or_path"], "| LoRA r =", ADAPTER_CONFIG.get("r"),
      "alpha =", ADAPTER_CONFIG.get("lora_alpha"))
print("Prompt contract:  ", json.dumps(CONTRACT))
print("Adapter files:    ", {k: v[:12] for k, v in ADAPTER_SHA_BEFORE.items()})
print("Phase 4 GPU:      ", TRAIN_REPORT.get("gpu"), "| this session:", GPU_NAME)
if TRAIN_REPORT.get("gpu", {}).get("name") != GPU_NAME:
    print("NOTE: a different GPU than Phase 4. Numerics (fp16 vs bf16) may differ slightly; recorded in the report.")
previous = sorted((Path(DRIVE_DIR) / "phase5").glob("*_frozen_eval")) if (Path(DRIVE_DIR) / "phase5").is_dir() else []
if previous:
    print(f"WARNING: frozen evaluation already run before ({[p.name for p in previous]}). A re-run must reproduce it; "
          "it must not be used to pick settings or runs.")

## 3. Evaluation bundle
The frozen set plus the exact Phase 4 plan format, scorer and backend (hash-checked by `make_eval_bundle.py`).

In [ ]:
if not os.path.isfile(EVAL_BUNDLE_ZIP):
    stop(f"Bundle not found at {EVAL_BUNDLE_ZIP}. Upload marketingiq_eval_bundle.zip to MyDrive/marketingiq_tiny/.")
shutil.rmtree(WORK_DIR, ignore_errors=True)
with zipfile.ZipFile(EVAL_BUNDLE_ZIP) as z:
    names = z.namelist()
    if any(n.endswith(("train.jsonl", "validation.jsonl")) for n in names):
        stop("The evaluation bundle contains training data; rebuild it with make_eval_bundle.py.")
    z.extractall(WORK_DIR)
BUNDLE_MANIFEST = json.loads((Path(WORK_DIR) / "bundle_manifest.json").read_text())
bad = [rel for rel, h in BUNDLE_MANIFEST.items() if sha256_file(Path(WORK_DIR) / rel) != h]
if bad:
    stop(f"Unzipped files differ from the bundle manifest: {bad}")
print(f"Unzipped {len(names)} files to {WORK_DIR}; all match bundle_manifest.json")
for n in names:
    print("  ", n)

## 4. Frozen evaluation integrity
Checked **before** a single question is parsed. Any mismatch stops the notebook.

In [ ]:
import sys
EVAL_CODE = Path(WORK_DIR) / "experiments" / "tiny_model" / "evaluation"
FROZEN_PATH = Path(WORK_DIR) / "experiments" / "tiny_model" / "dataset" / "eval_frozen.jsonl"
sys.path.insert(0, str(EVAL_CODE))
import frozen_eval as fe          # Phase 5 harness (imports score.py and plan_format.py from the bundle)
pf, scorer = fe.pf, fe.scorer

if fe.FROZEN_SHA256 != FROZEN_SHA256 or fe.FROZEN_COUNT != FROZEN_COUNT:
    stop("frozen_eval.py expects a different frozen set than section 0.")
try:
    FROZEN_DIGEST = fe.verify_frozen(FROZEN_PATH)
except fe.IntegrityError as e:
    stop(str(e))

## 5. Load questions, backend and scoring self-test
The prompt contract saved with the adapter must equal the bundled `plan_format.py`. The self-test scores every
**gold** plan as if it were a prediction (no model involved): it must be 429/429, otherwise the harness is broken.

In [ ]:
import collections
rows = fe.load_frozen(FROZEN_PATH)
try:
    pf.backend()
except Exception as e:
    stop(f"MarketingIQ backend modules could not load ({type(e).__name__}: {e}). Re-run section 1 (duckdb install).")

problems = []
if CONTRACT["system_prompt"] != pf.SYSTEM_PROMPT: problems.append("system prompt differs from Phase 4")
if CONTRACT["chat_template_kwargs"] != {"enable_thinking": False}: problems.append("thinking is not OFF in the contract")
if CONTRACT["stop_token"] != fe.EOS: problems.append(f"stop token {CONTRACT['stop_token']!r} != {fe.EOS!r}")
if not CONTRACT.get("add_generation_prompt"): problems.append("contract does not use add_generation_prompt")
if problems:
    stop("Prompt contract mismatch:\n" + "\n".join(problems))

fe.self_test(rows)
print(f"\n{len(rows)} questions | follow-ups (previous plan given): {sum(r['previous'] is not None for r in rows)}")
print("categories:", dict(collections.Counter(r["category"] for r in rows).most_common()))
print("gold intents:", dict(collections.Counter(r["plan"]["intent"] for r in rows).most_common()))

## 6. Load Qwen3-0.6B + the Phase 4 LoRA (inference only)
Unsloth loads the base model named in `adapter_config.json` with the same settings as Phase 4
(16-bit, `max_seq_length` from the report) and attaches the saved adapter. The live LoRA tensors are compared
with `adapter_model.safetensors`, then fingerprinted so section 8 can prove generation changed nothing.

In [ ]:
import torch
from unsloth import FastLanguageModel
from transformers import set_seed
set_seed(SEED)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = str(ADAPTER_DIR),     # adapter folder -> base model + LoRA (no merge, no quantization)
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = False,
    load_in_8bit = False,
    full_finetuning = False,
)
FastLanguageModel.for_inference(model)
model.eval()

loaded_base = model.peft_config["default"].base_model_name_or_path
if str(loaded_base).lower() != MODEL_ID.lower():
    stop(f"Loaded base model {loaded_base!r} is not {MODEL_ID!r}")
if tokenizer.eos_token != fe.EOS:
    stop(f"Unexpected eos token {tokenizer.eos_token!r}; expected {fe.EOS!r}")
try:
    LORA_INFO = fe.check_lora_loaded(model, ADAPTER_DIR)
except fe.IntegrityError as e:
    stop(str(e))
LORA_DIGEST_BEFORE = fe.lora_digest(model)
print(f"Base: {loaded_base} | dtype: {model.dtype} | LoRA tensors loaded: {LORA_INFO['lora_tensors']} "
      f"({LORA_INFO['lora_parameters']:,} parameters), identical to the saved adapter")
print(f"eos: {tokenizer.eos_token!r} | pad: {tokenizer.pad_token!r} | max_seq_length: {MAX_SEQ_LENGTH} | "
      f"max_new_tokens: {MAX_NEW_TOKENS}")

## 7. Prompts (exactly the Phase 4 format)
`plan_format.messages(question, previous)` → Qwen3 chat template with `enable_thinking=False` and the generation prompt.
Follow-ups get their `previous` plan through `plan_format.user_prompt()`, as in training. Every prompt is also rendered
with the Hub tokenizer of the base model and must be byte-identical (the same check Phase 4 made).

In [ ]:
from transformers import AutoTokenizer
import numpy as np
prompts = [fe.build_prompt(tokenizer, r) for r in rows]
hub_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
drift = [r["id"] for r, p in zip(rows, prompts) if fe.build_prompt(hub_tokenizer, r) != p]
if drift:
    stop(f"{len(drift)} prompts render differently with the Hub tokenizer, e.g. {drift[:3]}")
PROMPT_TOKENS = np.array([len(x) for x in tokenizer(prompts, add_special_tokens=False)["input_ids"]])
print(f"{len(prompts)} prompts, identical with the Hub chat template | prompt tokens max {PROMPT_TOKENS.max()}, "
      f"p50 {np.percentile(PROMPT_TOKENS, 50):.0f}")
if PROMPT_TOKENS.max() + MAX_NEW_TOKENS > MAX_SEQ_LENGTH:
    print(f"NOTE: the longest prompt + max_new_tokens exceeds the training max_seq_length ({MAX_SEQ_LENGTH}); "
          "nothing is truncated, recorded in the report.")
print("\nPrompt format (first follow-up):\n" + prompts[next(i for i, r in enumerate(rows) if r["previous"])])

## 8. ▶ Generate (greedy, thinking OFF)
`do_sample=False`, no temperature/top-p/top-k, stop at `<|im_end|>`, Phase 4 pad token, left padding, batch 32.
Only the newly generated tokens are decoded. A second pass at batch size 1 (no padding) checks that batching
did not change any output; it is a diagnostic only and does not replace the official outputs.

In [ ]:
t0 = time.time()
generations = fe.generate(model, tokenizer, prompts, MAX_NEW_TOKENS, GEN_BATCH_SIZE)
GEN_SECONDS = time.time() - t0
for r, g in zip(rows, generations):
    g["id"] = r["id"]
print(f"{len(generations)} plans in {GEN_SECONDS:.0f}s (GPU batch inference; not a CPU latency number)")

t0 = time.time()
single = fe.generate(model, tokenizer, prompts, MAX_NEW_TOKENS, 1)
PADDING_DIFF = [r["id"] for r, a, b in zip(rows, generations, single) if a["raw_output"] != b["raw_output"]]
print(f"batch size 1 check ({time.time() - t0:.0f}s): {len(PADDING_DIFF)} outputs differ from the batched run {PADDING_DIFF[:10]}")

LORA_DIGEST_AFTER = fe.lora_digest(model)
if LORA_DIGEST_AFTER != LORA_DIGEST_BEFORE:
    stop("LoRA weights changed during generation - results are invalid.")
print("LoRA weights unchanged by generation.")

## 9. Score
Per question: generation → parse → canonicalization → `validate_plan()` → `execute_plan()` (predicted and gold
plans, real DuckDB) → comparison by `score.py`. A crash inside `execute_plan()` (known backend bug) is an execution error.

In [ ]:
records = fe.score_all(rows, generations)
METRICS = fe.summarize(records)
P, A, NA = METRICS["pipeline"], METRICS["answerable"], METRICS["not_answerable"]
pct = fe._pct
print("PHASE 5 FINAL FROZEN EVALUATION (eval_frozen.jsonl, 429 questions)\n")
print("Exact Plan Match:      ", pct(METRICS["exact_plan_match"]))
print("Execution Match:       ", pct(METRICS["execution_match"]))
print("Strict JSON match:     ", pct(METRICS["exact_plan_match_strict_json"]))
print()
for k, v in P.items():
    print(f"{k.replace('_', ' '):24s}", pct(v))
print(f"\nAnswerable ({A['examples']}):")
for k in ("execution_attempted", "execution_successful", "execution_match", "execution_mismatch", "false_refusal"):
    print(f"  {k.replace('_', ' '):22s}", pct(A[k]))
print(f"  {'execution error':22s} {A['execution_error']['total']} of {A['execution_error']['of_attempted']} attempted")
print(f"\nNot answerable ({NA['examples']}): refused with the right reason", pct(NA["correctly_refused"]))
print("\nOutcomes:", METRICS["outcomes"])

In [ ]:
import pandas as pd
for g in fe.GROUPS:
    t = pd.DataFrame(METRICS["breakdowns"][g]).T
    t[["parsed", "validate_plan_ok", "exact", "execution_match"]] *= 100
    print(f"\nBy {g} (%):\n", t.round(1).to_string())
print("\nField accuracy on answerable questions:")
for f, m in A["field_accuracy"].items():
    print(f"  {f:20s} {pct(m)}")

## 10. Save results to Google Drive
Predictions (every stage per question), raw generations, report and a Markdown summary.

In [ ]:
ADAPTER_SHA_AFTER = hash_dir(ADAPTER_DIR)
if ADAPTER_SHA_AFTER != ADAPTER_SHA_BEFORE:
    stop("Adapter files changed during Phase 5 - results are invalid.")

REPORT5 = {
    "phase": 5, "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "training_performed": False,
    "frozen_eval": {"file": "eval_frozen.jsonl", "sha256": FROZEN_DIGEST, "examples": len(rows), "integrity": "PASS"},
    "model": {"base_model": loaded_base, "phase4_run": RUN_DIR.name, "adapter": "Phase 4 adapter/ (final model, no checkpoint selection)",
              "adapter_sha256": ADAPTER_SHA_BEFORE, "adapter_unchanged": True, "lora": LORA_INFO,
              "lora_weights_digest": LORA_DIGEST_BEFORE, "dtype": str(model.dtype), "merged": False, "quantized": False},
    "inference": {"prompt": "plan_format.messages() + Qwen3 chat template", "thinking": "OFF (enable_thinking=False)",
                  "do_sample": False, "max_new_tokens": MAX_NEW_TOKENS, "max_seq_length": MAX_SEQ_LENGTH,
                  "batch_size": GEN_BATCH_SIZE, "padding_side": "left", "eos_token": tokenizer.eos_token,
                  "pad_token": tokenizer.pad_token, "decoded": "new tokens only, skip_special_tokens, strip",
                  "prompt_tokens_max": int(PROMPT_TOKENS.max()), "generation_seconds": round(GEN_SECONDS, 1)},
    "padding_consistency": {"batch_size_1_outputs_differing": len(PADDING_DIFF), "ids": PADDING_DIFF},
    "phase4_validation_reference": {k: v for k, v in TRAIN_REPORT.items() if k.startswith("validation_") and k != "validation_per_category_pct"},
    "environment": {"gpu": GPU_NAME, "versions": VERSIONS, "phase4_gpu": TRAIN_REPORT.get("gpu"),
                    "phase4_versions": TRAIN_REPORT.get("versions")},
    "code_sha256": BUNDLE_MANIFEST,
    "scoring": "evaluation/score.py (exact, execution_match) via evaluation/frozen_eval.py; execute_plan crashes = execution errors",
    "metrics": METRICS,
    "groq_calls": 0,
}
OUT = Path(OUTPUT_DIR)
shutil.rmtree(OUT, ignore_errors=True)
fe.write_outputs(OUT, records, REPORT5)
with (OUT / "frozen_eval_generations.jsonl").open("w", encoding="utf-8") as fh:
    for g in generations:
        fh.write(json.dumps(g) + "\n")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
DRIVE_OUT = Path(DRIVE_DIR) / "phase5" / f"{stamp}_frozen_eval"
shutil.copytree(OUT, DRIVE_OUT)
bad = [p for p in OUT.rglob("*") if p.is_file() and sha256_file(p) != sha256_file(DRIVE_OUT / p.relative_to(OUT))]
if bad:
    stop(f"Copy to Drive failed for {bad}")
print("Saved to Google Drive:", DRIVE_OUT)
for p in sorted(DRIVE_OUT.iterdir()):
    print(f"{p.stat().st_size / 1e3:9.1f} KB  {p.name}")
print("\n" + (OUT / "frozen_eval_summary.md").read_text()[:1500])

## Phase 5 ends here
The numbers above are the final, unbiased result of the Phase 4 model on the frozen set. Do not retrain, re-prompt
or pick another checkpoint because of them. Export/quantization, local llama.cpp inference, latency/RAM benchmarks
and production integration are later phases.